In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import uuid
from scipy.stats import linregress
import pandas as pd
import logging
from datetime import datetime
from multiprocessing import Pool, Manager
from functools import partial



In [2]:
# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

def setup_process_logging(log_queue):
    """Configure logging for a multiprocessing worker to send logs to the main process."""
    logger = logging.getLogger(__name__)
    logger.handlers = []  # Clear default handlers
    queue_handler = logging.handlers.QueueHandler(log_queue)
    logger.addHandler(queue_handler)
    logger.setLevel(logging.INFO)
    return logger

In [3]:
# Constants
MAX_DAYS = 365  # For DaysSinceLastPurchase for cold SKUs
ROLLING_WINDOWS = [4, 8, 12]  # Multiple rolling window sizes
NEGATIVE_SAMPLING_RATIO = 2  # Negative to positive sample ratio
NUM_WEEKS = 21  # Number of historical weeks for training
INFERENCE_WEEK = 22  # Week for inference

CURRENT_DATE = datetime.now()

In [4]:

# from datetime import datetime, timedelta

# # CURRENT_DATE = datetime.now()  # or your specific CURRENT_DATE, e.g., datetime(2025, 1, 1)
# result_date = (CURRENT_DATE - timedelta(weeks=NUM_WEEKS - 1)).strftime('%Y-%m-%d')
# result_date
# pd.to_datetime(transactions['LastDeliveryDate']) #.strftime('%Y-%m-%d')

### UTILITIES

In [5]:
# Generate synthetic data  
def generate_synthetic_data():
    skus = pd.DataFrame({
        'SKUID': [f'SKU{i}' for i in range(1, 101)],
        'Manufacturer': np.random.choice(['M1', 'M2', 'M3'], 100),
        'Category': np.random.choice(['Cat1', 'Cat2', 'Cat3'], 100),
        'Segment': np.random.choice(['Seg1', 'Seg2'], 100)
    })
    
    customers = pd.DataFrame({
        'CustomerID': [f'C{i}' for i in range(1, 51)],
        'State': np.random.choice(['State1', 'State2'], 50),
        'City': np.random.choice(['City1', 'City2'], 50),
        'Town': np.random.choice(['Town1', 'Town2'], 50),
        'Recency': np.random.randint(1, 100, 50),
        'Frequency': np.random.randint(1, 50, 50),
        'Monetary': np.random.uniform(100, 10000, 50)
    })
    
    transactions = []
    start_date = datetime(2025, 1, 1)
    for week in range(1, NUM_WEEKS + 1):
        week_date = start_date + timedelta(weeks=week - 1)
        for _ in range(200):
            customer = np.random.choice(customers['CustomerID'])
            sku = np.random.choice(skus['SKUID'])
            transactions.append({
                'Week': week,
                'CustomerID': customer,
                'SKUID': sku,
                'OrderValue': np.random.uniform(10, 500),
                'TransactionDate': week_date
            })
    transactions = pd.DataFrame(transactions)
    
    return skus, customers, transactions


In [6]:
# Updated feature engineering function with multiple rolling windows
def compute_features(week, transactions, skus, customers, is_inference=False):
    if is_inference:
        hist_transactions = transactions[transactions['Week'] <= week - 1]
    else:
        hist_transactions = transactions[transactions['Week'] < week]
    
    features_list = []
    
    for _, row in transactions[transactions['Week'] == week].iterrows() if not is_inference else transactions.iterrows():
        customer_id = row['CustomerID'] if not is_inference else row['CustomerID']
        sku_id = row['SKUID'] if not is_inference else row['SKUID']
        feature_dict = {'Week': week, 'CustomerID': customer_id, 'SKUID': sku_id}
        
        sku_info = skus[skus['SKUID'] == sku_id].iloc[0] if sku_id in skus['SKUID'].values else None
        customer_info = customers[customers['CustomerID'] == customer_id].iloc[0]
        
        if sku_info is None:
            continue
        
        # Customer-SKU Interaction Features
        customer_sku_transactions = hist_transactions[
            (hist_transactions['CustomerID'] == customer_id) & 
            (hist_transactions['SKUID'] == sku_id)
        ]
        
        feature_dict['DaysSinceLastPurchase_SKU'] = (
            MAX_DAYS + 1 if customer_sku_transactions.empty 
            else (CURRENT_DATE + timedelta(weeks=week - 1) - 
                  customer_sku_transactions['LastDeliveryDate'].max()).days
        )
        
        feature_dict['TotalPurchases_SKU'] = len(customer_sku_transactions)
        feature_dict['AvgOrderValue_SKU_by_Customer'] = (
            customer_sku_transactions['OrderValue'].mean() if not customer_sku_transactions.empty else 0
        )
        
        # Customer General Behavior (RFM)
        feature_dict['Customer_Recency'] = customer_info['Recency']
        feature_dict['Customer_Frequency'] = customer_info['Frequency']
        feature_dict['Customer_Monetary'] = customer_info['Monetary']
        feature_dict['Customer_TotalUniqueSKUsPurchased_Overall'] = len(
            hist_transactions[hist_transactions['CustomerID'] == customer_id]['SKUID'].unique()
        )
        
        # Localized SKU Performance (State)
        state = customer_info['State'] ### -----> MODIFICATION HERE <-----
        state_transactions = hist_transactions[hist_transactions['CustomerID'].isin(
            customers[customers['State'] == state]['CustomerID'])]
        
        feature_dict['SKU_TotalSales_CustomerState_Overall'] = state_transactions[
                    state_transactions['SKUID'] == sku_id]['OrderValue'].sum()
        
        # Time-Series & Rolling Window (Multiple Windows: 4, 8, 12 weeks)
        for window in ROLLING_WINDOWS:
            rolling_transactions = hist_transactions[
                (hist_transactions['Week'] >= week - window) & 
                (hist_transactions['Week'] <= week - 1)
            ]
            customer_rolling = rolling_transactions[rolling_transactions['CustomerID'] == customer_id]
            
            # Rolling features for each window
            feature_dict[f'Customer_RollingAvgOrderValue_{window}Weeks'] = (
                customer_rolling['OrderValue'].mean() if not customer_rolling.empty else 0
            )
            feature_dict[f'Customer_RollingUniqueSKUsPurchased_{window}Weeks'] = len(customer_rolling['SKUID'].unique())
            feature_dict[f'Customer_RollingPurchaseCount_{window}Weeks'] = len(customer_rolling)
            
            # SKU-specific rolling features
            sku_rolling = rolling_transactions[rolling_transactions['SKUID'] == sku_id]
            feature_dict[f'SKU_RollingTotalSales_{window}Weeks'] = sku_rolling['OrderValue'].sum()
            feature_dict[f'SKU_RollingPurchaseCount_{window}Weeks'] = len(sku_rolling)
            
            # Category-specific rolling features
            category = sku_info['CategoryID']
            category_rolling = rolling_transactions[rolling_transactions['SKUID'].isin(
                skus[skus['CategoryID'] == category]['SKUID']
            )]
            feature_dict[f'Category_RollingTotalSales_{window}Weeks'] = category_rolling['OrderValue'].sum()
            
        # Trend-based feature: Slope of Customer_RollingPurchaseCount over weeks
        rolling_weeks = hist_transactions[
            (hist_transactions['Week'] >= week - max(ROLLING_WINDOWS)) & 
            (hist_transactions['Week'] <= week - 1) &
            (hist_transactions['CustomerID'] == customer_id)
        ]
        if not rolling_weeks.empty:
            week_counts = rolling_weeks.groupby('Week').size().reset_index(name='PurchaseCount')
            if len(week_counts) > 1:
                slope, _, _, _, _ = linregress(week_counts['Week'], week_counts['PurchaseCount'])
                feature_dict['Customer_PurchaseCount_Trend'] = slope
            else:
                feature_dict['Customer_PurchaseCount_Trend'] = 0
        else:
            feature_dict['Customer_PurchaseCount_Trend'] = 0
        
        # Granular Location Affinity (Town)
        town = customer_info['Town']
        town_transactions = hist_transactions[hist_transactions['CustomerID'].isin(
            customers[customers['Town'] == town]['CustomerID'])]
        
        for window in ROLLING_WINDOWS:
            town_window_transactions = town_transactions[
                (town_transactions['Week'] >= week - window) & 
                (town_transactions['Week'] <= week - 1)
            ]
            feature_dict[f'Town_SKU_PurchaseCount_{window}Weeks'] = len(
                town_window_transactions[town_window_transactions['SKUID'] == sku_id]
            )
        
        features_list.append(feature_dict)
    
    return pd.DataFrame(features_list)


In [7]:
# Generate training data 
def generate_training_data_(skus, customers, transactions):
    all_training_data = []
    
    for week in range(1, NUM_WEEKS + 1):
        week_transactions = transactions[transactions['Week'] == week][['Week', 'CustomerID', 'SKUID']].copy()
        if not week_transactions.empty:
            week_transactions['purchase_likelihood'] = 1
        
            negative_samples = []
            for customer_id in week_transactions['CustomerID'].unique():
                purchased_skus = week_transactions[week_transactions['CustomerID'] == customer_id]['SKUID'].unique()
                non_purchased_skus = skus[~skus['SKUID'].isin(purchased_skus)]['SKUID'].sample(
                    n=len(purchased_skus) * NEGATIVE_SAMPLING_RATIO
                )
                for sku_id in non_purchased_skus:
                    negative_samples.append({
                        'Week': week,
                        'CustomerID': customer_id,
                        'SKUID': sku_id,
                        'purchase_likelihood': 0
                    })
            
            week_data = pd.concat([
                week_transactions,
                pd.DataFrame(negative_samples)
            ], ignore_index=True)
            
            week_features = compute_features(week, transactions, skus, customers)
            week_data = week_data.merge(week_features, on=['Week', 'CustomerID', 'SKUID'], how='left')
            
            all_training_data.append(week_data)
    
    return pd.concat(all_training_data, ignore_index=True)


In [ ]:
def process_week(week, skus, customers, transactions, transactions_with_state, 
                 negative_sampling_ratio, max_negatives, look_back_weeks, log_queue):
    """
    Process data for a single week, including transactions, negative sampling, and feature computation.

    Parameters:
    - week (int): Week number to process.
    - skus (pd.DataFrame): DataFrame with SKUID column.
    - customers (pd.DataFrame): DataFrame with CustomerID and State columns.
    - transactions (pd.DataFrame): DataFrame with Week, CustomerID, SKUID columns.
    - transactions_with_state (pd.DataFrame): Precomputed transactions with State column.
    - negative_sampling_ratio (int): Ratio of negative to positive samples.
    - max_negatives (int): Maximum number of negative samples per customer.
    - look_back_weeks (int): Number of past weeks for negative sampling.
    - log_queue (Queue): Queue for logging across processes.

    Returns:
    - pd.DataFrame or None: Processed data for the week or None if no transactions.
    """
    logger = setup_process_logging(log_queue)
    logger.info("Processing week %d", week)
    
    # Filter transactions for the current week
    week_transactions = transactions[transactions['Week'] == week][['Week', 'CustomerID', 'SKUID']].copy()
    logger.info("Found %d transactions for week %d", len(week_transactions), week)
    
    if week_transactions.empty:
        logger.warning("No transactions found for week %d, skipping", week)
        return None
    
    # Assign purchase likelihood for positive samples
    week_transactions['purchase_likelihood'] = 1
    logger.debug("Assigned purchase_likelihood=1 to %d positive samples", len(week_transactions))
    
    # Generate negative samples
    negative_samples = []
    unique_customers = week_transactions['CustomerID'].unique()
    logger.info("Generating negative samples for %d unique customers", len(unique_customers))
    
    for i, customer_id in enumerate(unique_customers, 1):
        # Get customer's state
        try:
            customer_state = customers[customers['CustomerID'] == customer_id]['State'].iloc[0]
        except IndexError:
            logger.error("Customer %s not found in customers DataFrame, skipping", customer_id)
            continue
        
        # Get purchased SKUs for this customer in this week
        purchased_skus = week_transactions[week_transactions['CustomerID'] == customer_id]['SKUID'].unique()
        
        # Get SKUs available in the customer's state from current and past weeks
        state_skus = transactions_with_state[
            (transactions_with_state['State'] == customer_state) &
            (transactions_with_state['Week'].between(max(1, week - look_back_weeks), week))
        ]['SKUID'].unique()
        
        # Get non-purchased SKUs
        non_purchased_skus = pd.Series(state_skus)[~pd.Series(state_skus).isin(purchased_skus)]
        
        # Calculate number of negative samples (hybrid capped proportional)
        n_samples = min(len(purchased_skus) * negative_sampling_ratio, max_negatives)
        
        try:
            if non_purchased_skus.empty:
                logger.warning("No non-purchased SKUs for customer %s in state %s, week %d. Falling back to all SKUs.", 
                               customer_id, customer_state, week)
                non_purchased_skus = skus['SKUID'][~skus['SKUID'].isin(purchased_skus)]
                if non_purchased_skus.empty:
                    logger.error("No non-purchased SKUs available even in fallback for customer %s, week %d", customer_id, week)
                    continue
            sampled_skus = non_purchased_skus.sample(n=n_samples, replace=True)
            for sku_id in sampled_skus:
                negative_samples.append({
                    'Week': week,
                    'CustomerID': customer_id,
                    'SKUID': sku_id,
                    'purchase_likelihood': 0
                })
            if i % max(1, len(unique_customers) // 5) == 0:
                logger.debug("Processed %d/%d customers for negative sampling in week %d", i, len(unique_customers), week)
        except ValueError as e:
            logger.error("Error sampling non-purchased SKUs for customer %s in week %d: %s", customer_id, week, str(e))
            continue
    
    logger.info("Generated %d negative samples for week %d", len(negative_samples), week)
    
    # Combine positive and negative samples
    week_data = pd.concat([week_transactions, pd.DataFrame(negative_samples)], ignore_index=True)
    logger.debug("Combined %d total samples for week %d", len(week_data), week)
    
    # Compute features
    logger.debug("Computing features for week %d", week)
    try:
        week_features = compute_features(week, transactions, skus, customers)
        logger.debug("Computed features with %d rows for week %d", len(week_features), week)
        
        # Merge features with week data
        week_data = week_data.merge(week_features, on=['Week', 'CustomerID', 'SKUID'], how='left')
        logger.info("Merged features, resulting in %d rows for week %d", len(week_data), week)
    except Exception as e:
        logger.error("Error computing/merging features for week %d: %s", week, str(e))
        return None
    
    return week_data


### GENERATE TRAINING DATA

In [9]:

def generate_training_data(skus, customers, transactions, min_week=20,max_week=NUM_WEEKS, 
                           negative_sampling_ratio=1, max_negatives=15, 
                           look_back_weeks=4):
    """
    Generate training data with positive and negative samples for each week, using parallel processing.

    Parameters:
    - skus (pd.DataFrame): DataFrame with SKUID column.
    - customers (pd.DataFrame): DataFrame with CustomerID and State columns.
    - transactions (pd.DataFrame): DataFrame with Week, CustomerID, SKUID columns.
    - num_weeks (int): Number of weeks to process.
    - negative_sampling_ratio (int): Ratio of negative to positive samples (default: 1).
    - max_negatives (int): Maximum number of negative samples per customer (default: 15).
    - look_back_weeks (int): Number of past weeks for negative sampling (default: 4).

    Returns:
    - pd.DataFrame: Combined training data with features and purchase likelihood.
    """
    logger.info("Starting training data generation for %d weeks with %d-week look-back", 
                max_week - min_week + 1, look_back_weeks)
    
    # Precompute transactions with state to reduce overhead in each process
    logger.debug("Precomputing transactions with state information")
    try:
        transactions_with_state = transactions.merge(
            customers[['CustomerID', 'State']],
            on='CustomerID',
            how='left'
        )
        logger.debug("Precomputed transactions_with_state with %d rows", len(transactions_with_state))
    except Exception as e:
        logger.error("Error precomputing transactions_with_state: %s", str(e))
        return pd.DataFrame()
    
    # Set up multiprocessing with a shared logging queue
    manager = Manager()
    log_queue = manager.Queue()
    
    # Process weeks in parallel
    process_week_partial = partial(
        process_week,
        skus=skus,
        customers=customers,
        transactions=transactions,
        transactions_with_state=transactions_with_state,
        negative_sampling_ratio=negative_sampling_ratio,
        max_negatives=max_negatives,
        look_back_weeks=look_back_weeks,
        log_queue=log_queue
    )
    
    with Pool(processes=4) as pool:  # Limit to 4 processes to manage resource usage
        results = pool.map(process_week_partial, range(min_week, max_week + 1))
    
    # Process logs from the queue
    while not log_queue.empty():
        record = log_queue.get()
        logger.handle(record)
    
    # Filter out None results (failed or skipped weeks)
    all_training_data = [result for result in results if result is not None]
    logger.info("Combining data from %d valid weeks", len(all_training_data))
    
    # Combine all weeks
    final_data = pd.concat(all_training_data, ignore_index=True) if all_training_data else pd.DataFrame()
    logger.info("Training data generation complete. Total rows: %d", len(final_data))
    
    return final_data

### GENERATE INFERENCE DATA

In [ ]:
# Generate inference data  
def generate_inference_data___(skus, customers, transactions): 
    
    inference_samples = []
    for customer_id in customers['CustomerID']:
        purchased_skus = transactions[transactions['CustomerID'] == customer_id]['SKUID'].unique()
        customer_state = customers[customers['CustomerID'] == customer_id]['State'].iloc[0]
        state_customers = customers[customers['State'] == customer_state]['CustomerID']
        state_transactions = transactions[transactions['CustomerID'].isin(state_customers)]
        popular_skus = state_transactions.groupby('SKUID')['OrderValue'].sum().nlargest(5).index
        customer_categories = transactions[transactions['CustomerID'] == customer_id].merge(skus, on='SKUID')['Category'].value_counts().index[:2]
        category_skus = skus[skus['Category'].isin(customer_categories)]['SKUID'].sample(5, replace=True)
        curated_skus = list(set(list(purchased_skus) + list(popular_skus) + list(category_skus)))
        
        for sku_id in curated_skus[:50]:
            inference_samples.append({
                'Week': INFERENCE_WEEK,
                'CustomerID': customer_id,
                'SKUID': sku_id
            })
    
    inference_data = pd.DataFrame(inference_samples)
    inference_features = compute_features(INFERENCE_WEEK, transactions, skus, customers, is_inference=True)
    inference_data = inference_data.merge(inference_features, on=['Week', 'CustomerID', 'SKUID'], how='left')
    
    return inference_data

In [ ]:
import pandas as pd
import numpy as np
import multiprocessing
from functools import partial

def process_customer_chunk(customer_chunk, top_popular_skus_per_state, top_skus_per_state_category, 
                         purchased_skus_per_customer, customer_categories, skus, INFERENCE_WEEK):
    inference_samples = []
    for customer_id in customer_chunk:
        # Get customer state
        customer_row = customers[customers['CustomerID'] == customer_id]
        if customer_row.empty:
            continue
        customer_state = customer_row['State'].iloc[0]
        
        # Get purchased SKUs
        purchased_skus = purchased_skus_per_customer.get(customer_id, [])
        
        # Get popular SKUs in the state
        popular_skus = top_popular_skus_per_state.get(customer_state, [])
        
        # Get customer's purchased categories
        customer_categories_list = customer_categories.get(customer_id, [])
        
        # Get top 5 SKUs from customer's categories in the state
        if customer_categories_list:
            category_top_skus_df = top_skus_per_state_category[
                (top_skus_per_state_category['State'] == customer_state) & 
                (top_skus_per_state_category['Category'].isin(customer_categories_list))
            ].sort_values('OrderValue', ascending=False)
            category_top_skus = category_top_skus_df['SKUID'].head(5).tolist()
        else:
            category_top_skus = []
        
        # Combine SKUs, remove duplicates while preserving order
        curated_skus = list(dict.fromkeys(list(purchased_skus) + list(popular_skus) + list(category_top_skus)))
        
        # Fallback if curated_skus is empty
        if not curated_skus:
            curated_skus = skus['SKUID'].sample(5, replace=True).tolist()
        
        # Create inference samples with up to 50 SKUs
        for sku_id in curated_skus[:50]:
            inference_samples.append({
                'Week': INFERENCE_WEEK,
                'CustomerID': customer_id,
                'SKUID': sku_id
            })
    return inference_samples

def generate_inference_data(skus, customers, transactions, INFERENCE_WEEK):
    """
    Generate inference data with optimized customer selection and SKU sampling.
    
    Parameters:
    - skus: DataFrame with SKU information
    - customers: DataFrame with customer information
    - transactions: DataFrame with transaction history
    - INFERENCE_WEEK: Integer representing the week for inference
    
    Returns:
    - inference_data: DataFrame with inference samples and features
    """
    # Precompute transactions with state
    transactions_with_state = transactions.merge(customers[['CustomerID', 'State']], 
                                               on='CustomerID', how='left')
    
    # Precompute top 5 popular SKUs per state
    state_sku_popularity = transactions_with_state.groupby(['State', 'SKUID'])['OrderValue']\
        .sum().reset_index()
    top_popular_skus_per_state = state_sku_popularity\
        .sort_values(['State', 'OrderValue'], ascending=[True, False])\
        .groupby('State').head(5).groupby('State')['SKUID'].apply(list).to_dict()
    
    # Precompute top 10 SKUs per state per category
    state_category_sku_popularity = transactions_with_state.merge(skus[['SKUID', 'Category']], 
                                                               on='SKUID')\
        .groupby(['State', 'Category', 'SKUID'])['OrderValue'].sum().reset_index()
    top_skus_per_state_category = state_category_sku_popularity\
        .sort_values(['State', 'Category', 'OrderValue'], ascending=[True, True, False])\
        .groupby(['State', 'Category']).head(10)
    
    # Precompute purchased SKUs and categories per customer
    purchased_skus_per_customer = transactions.groupby('CustomerID')['SKUID'].unique().to_dict()
    customer_categories = transactions.merge(skus[['SKUID', 'Category']], on='SKUID')\
        .groupby('CustomerID')['Category'].unique().to_dict()
    
    # Select active customers (last 6 weeks)
    active_weeks = list(range(max(1, INFERENCE_WEEK - 6), INFERENCE_WEEK))
    active_customers = transactions[transactions['Week'].isin(active_weeks)]['CustomerID'].unique()
    
    # Randomly sample 20% of other customers
    all_customers = customers['CustomerID'].unique()
    other_customers = [c for c in all_customers if c not in active_customers]
    sample_size = int(0.2 * len(active_customers))
    if sample_size > 0 and len(other_customers) > 0:
        sampled_other_customers = np.random.choice(other_customers, 
                                                 size=min(sample_size, len(other_customers)), 
                                                 replace=False)
    else:
        sampled_other_customers = []
    
    # Combine selected customers
    selected_customers = list(active_customers) + list(sampled_other_customers)
    
    # Parallel processing
    num_processes = multiprocessing.cpu_count()
    chunk_size = max(1, len(selected_customers) // num_processes)
    customer_chunks = [selected_customers[i:i + chunk_size] 
                      for i in range(0, len(selected_customers), chunk_size)]
    
    with multiprocessing.Pool(num_processes) as pool:
        process_func = partial(process_customer_chunk,
                             top_popular_skus_per_state=top_popular_skus_per_state,
                             top_skus_per_state_category=top_skus_per_state_category,
                             purchased_skus_per_customer=purchased_skus_per_customer,
                             customer_categories=customer_categories,
                             skus=skus,
                             INFERENCE_WEEK=INFERENCE_WEEK)
        results = pool.map(process_func, customer_chunks)
    
    # Combine results
    inference_samples = [item for sublist in results for item in sublist]
    inference_data = pd.DataFrame(inference_samples)
    
    # Compute and merge features
    inference_features = compute_features(INFERENCE_WEEK, transactions, skus, customers, 
                                        is_inference=True)
    inference_data = inference_data.merge(inference_features, 
                                        on=['Week', 'CustomerID', 'SKUID'], 
                                        how='left')
    
    return inference_data

# Note: compute_features is assumed to be an existing function

## MANUAL RUN

In [11]:
# skus, customers, transactions = generate_synthetic_data()
# skus.to_csv('./data/skus.csv', index=False)
# customers.to_csv('./data/customers.csv', index=False)
# transactions.to_csv('./data/transactions.csv', index=False) 

In [39]:
PATH_DIM_SKU = './data/dim_sku.csv'
PATH_DIM_CUSTOMER = './data/dim_customer.csv'
PATH_FACT_TRANSACTION = './data/fact_transaction.csv'

skus = pd.read_csv(PATH_DIM_SKU).rename(columns={'CategoryName':'Category'})
customers = pd.read_csv(PATH_DIM_CUSTOMER).rename(columns={'StateName':'State',
                                                           'CityName':'City',
                                                           'TownName':'Town'})  
transactions = pd.read_csv(PATH_FACT_TRANSACTION).rename(columns={'lastDeliveryDate':'LastDeliveryDate'})
transactions['LastDeliveryDate'] = pd.to_datetime(transactions['LastDeliveryDate'])



/tmp/ipykernel_72140/2537260605.py:6: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  customers = pd.read_csv(PATH_DIM_CUSTOMER).rename(columns={'StateName':'State',


In [18]:
print(sorted(transactions.Week.unique().tolist()))

[13, 14, 15, 16, 17, 18, 19, 20, 21, 22]


In [28]:
print(skus.columns.tolist())
print(customers.columns.tolist())
print(transactions.columns.tolist())

['SKUID', 'SKUCODE', 'ProductName', 'CategoryID', 'Category', 'Segment', 'BrandID', 'BrandName', 'ManufacturerID', 'ManufacturerName']
['CustomerID', 'Recency', 'Frequency', 'Monetary', 'BusinessID', 'ContactName', 'ContactPhone', 'StateID', 'State', 'CityID', 'City', 'TownID', 'Town', 'Latitude', 'Longitude', 'CustomerTypeID', 'CustomerModeName', 'CustomerType', 'CustomerStatus']
['Week', 'CustomerID', 'SKUID', 'SKUCODE', 'OrderCount', 'OrderFreqDays', 'OrderQty', 'OrderValue', 'LastDeliveryDate']


In [ ]:
# training_data =  generate_training_data(skus, customers, transactions, 
#                                         min_week = 13, max_week = 21,  
#                                         negative_sampling_ratio=1, max_negatives=15, 
#                                         look_back_weeks=4
#             )
# # print(training_data)

2025-05-29 21:05:27 - INFO - Starting training data generation for 9 weeks with 4-week look-back
2025-05-29 21:05:28 - INFO - Processing week 13
2025-05-29 21:05:28 - INFO - Found 594 transactions for week 13
2025-05-29 21:05:28 - INFO - Generating negative samples for 189 unique customers
2025-05-29 21:05:29 - INFO - Processing week 14
2025-05-29 21:05:29 - INFO - Found 12262 transactions for week 14
2025-05-29 21:05:29 - INFO - Generating negative samples for 3309 unique customers
2025-05-29 21:05:30 - INFO - Processing week 15
2025-05-29 21:05:30 - INFO - Found 13564 transactions for week 15
2025-05-29 21:05:30 - INFO - Generating negative samples for 3907 unique customers
2025-05-29 21:05:30 - INFO - Processing week 16
2025-05-29 21:05:30 - INFO - Found 10618 transactions for week 16
2025-05-29 21:05:30 - INFO - Generating negative samples for 3193 unique customers
2025-05-29 21:05:30 - INFO - Generated 594 negative samples for week 13
2025-05-29 21:05:33 - WARNING - No non-purchas

In [40]:
# training_data.to_feather('./input/training_data.feather', compression='zstd')

In [ ]:
# training_data.head(100)
# print(training_data.shape)
# training_data.Week.unique().tolist()

(177235, 34)


In [ ]:
# customers.head()
# skus.head()

# transactions[['Week']].value_counts().sort_index()
# week_transactions = transactions[transactions['Week'] == 1][['Week', 'CustomerID', 'SKUID']].copy()

In [46]:
transactions.Week.unique()

array([20, 21, 17, 18, 15, 16, 14, 19, 22, 13])

In [42]:
customers.head(1)

,CustomerID,Recency,Frequency,Monetary,BusinessID,ContactName,ContactPhone,StateID,State,CityID,City,TownID,Town,Latitude,Longitude,CustomerTypeID,CustomerModeName,CustomerType,CustomerStatus
0,5144015,0,0,0,76,Ayobami Akangbe,8064534198,131.0,Oyo,1641.0,Ibadan North East,21132.0,Gate,7.3919442,3.961699,2,Retailer,Customer,Approved


In [41]:
inference_data = generate_inference_data(skus, customers, transactions)

KeyboardInterrupt: 

## AUTO RUN

In [ ]:
# Run the data generation
if __name__ == "__main__": 
    # skus, customers, transactions = generate_synthetic_data()
    
    PATH_DIM_SKU = './data/dim_sku.csv'
    PATH_DIM_CUSTOMER = './data/dim_customer.csv'
    PATH_FACT_TRANSACTION = './data/fact_transaction.csv'
    
    skus = pd.read_csv(PATH_DIM_SKU)
    customers = pd.read_csv(PATH_DIM_CUSTOMER) 
    transactions = pd.read_csv(PATH_FACT_TRANSACTION).rename(columns={'lastDeliveryDate':'LastDeliveryDate'})  
    
    training_data = generate_training_data(skus, customers, transactions)
    training_data.to_csv('input/training_data_multi_windows.csv', index=False)
    print("Training data with multiple rolling windows generated and saved to 'input/training_data_multi_windows.csv'")
    
    inference_data = generate_inference_data(skus, customers, transactions)
    inference_data.to_csv('input/inference_data_multi_windows.csv', index=False)
    print("Inference data with multiple rolling windows generated and saved to 'input/inference_data_multi_windows.csv'")

In [ ]:
print("transactions Unique Weeks: ", transactions.Week.unique().tolist())
print("training_data Unique Weeks: ", training_data.Week.unique().tolist())
print("inference_data Unique Weeks: ", inference_data.Week.unique().tolist())
print(f"{'=' * 100}")
print("transactions Unique Weeks Count: ", transactions.Week.nunique())
print("training_data Unique Weeks Count: ", training_data.Week.nunique())
print("inference_data Unique Weeks Count: ", inference_data.Week.nunique())
print(f"{'=' * 100}")
print("transactions Unique Customers: ", transactions.CustomerID.nunique())
print("training_data Unique Customers: ", training_data.CustomerID.nunique())
print("inference_data Unique Customers: ", inference_data.CustomerID.nunique())
print(f"{'=' * 100}") 
print("training_data Features: ", training_data.columns.nunique())
print("inference_data Features: ", inference_data.columns.nunique())
print(f"{'=' * 100}") 
print("training_data Features: ", training_data.drop('purchase_likelihood', axis=1).columns.tolist())
print("inference_data Features: ", inference_data.columns.tolist())

## DASK IMPLEMENTATION

In [33]:
import dask.dataframe as dd
from dask import delayed
import pandas as pd

def compute_negative_samples(customer_id, week, transactions_with_state_dd, skus_dd, negative_sampling_ratio=1, max_negatives=15, look_back_weeks=4):
    """Generate negative samples for a customer-week combination."""
    # Filter transactions for the customer within the look-back period
    prior_weeks = transactions_with_state_dd[
        (transactions_with_state_dd['CustomerID'] == customer_id) &
        (transactions_with_state_dd['Week'].between(week - look_back_weeks, week - 1))
    ]
    purchased_skus = prior_weeks['SKUID'].unique().compute()
    
    # Get SKUs not purchased by the customer
    available_skus = skus_dd[~skus_dd['SKUID'].isin(purchased_skus)]['SKUID'].compute()
    num_negatives = min(int(negative_sampling_ratio * len(purchased_skus)), max_negatives, len(available_skus))
    
    # Sample negative SKUs
    if num_negatives > 0:
        negative_skus = available_skus.sample(n=num_negatives, random_state=week + customer_id)
        negative_df = pd.DataFrame({
            'Week': week,
            'CustomerID': customer_id,
            'SKUID': negative_skus,
            'purchase_likelihood': 0
        })
        return negative_df
    return pd.DataFrame()

def compute_features(week, transactions_dd, skus_dd, customers_dd):
    """Compute features for a given week (simplified for Dask compatibility)."""
    prior_transactions = transactions_dd[transactions_dd['Week'] < week]
    features = prior_transactions.groupby(['CustomerID', 'SKUID']).size().reset_index(name='purchase_count')
    return features

def process_week(week, skus_dd, customers_dd, transactions_dd, negative_samples_dd, log_queue=None):
    """Process data for a single week using Dask."""
    # Filter transactions for the current week (positive samples)
    week_transactions = transactions_dd[transactions_dd['Week'] == week][['Week', 'CustomerID', 'SKUID']].copy()
    week_transactions['purchase_likelihood'] = 1
    
    # Get precomputed negative samples for this week
    week_negative_samples = negative_samples_dd[negative_samples_dd['Week'] == week]
    
    # Combine positive and negative samples
    week_data = dd.concat([week_transactions, week_negative_samples], ignore_index=True)
    
    # Compute features and merge
    week_features = compute_features(week, transactions_dd, skus_dd, customers_dd)
    week_data = week_data.merge(week_features, on=['Week', 'CustomerID', 'SKUID'], how='left')
    
    # Materialize the result
    return week_data.compute()

def generate_training_data(skus, customers, transactions, num_weeks, negative_sampling_ratio=1, max_negatives=15, look_back_weeks=4):
    """Generate training data using Dask for memory efficiency."""
    # Convert Pandas DataFrames to Dask DataFrames
    transactions_dd = dd.from_pandas(transactions, npartitions=10)
    customers_dd = dd.from_pandas(customers, npartitions=5)
    skus_dd = dd.from_pandas(skus, npartitions=5)
    
    # Precompute transactions with state
    transactions_with_state_dd = transactions_dd.merge(
        customers_dd[['CustomerID', 'State']], on='CustomerID', how='left'
    )
    
    # Precompute negative samples in parallel
    unique_customers = transactions_dd['CustomerID'].unique().compute()
    negative_samples_tasks = [
        delayed(compute_negative_samples)(cid, w, transactions_with_state_dd, skus_dd, negative_sampling_ratio, max_negatives, look_back_weeks)
        for w in range(1, num_weeks + 1)
        for cid in unique_customers
    ]
    negative_samples_dd = dd.from_delayed(negative_samples_tasks)
    
    # Process weeks in parallel
    week_tasks = [
        delayed(process_week)(w, skus_dd, customers_dd, transactions_dd, negative_samples_dd)
        for w in range(1, num_weeks + 1)
    ]
    results = dd.from_delayed(week_tasks)
    
    # Compute and return the final result
    final_data = results.compute()
    return final_data

In [ ]:
# Example usage
if __name__ == "__main__":
    # Sample input data (replace with your actual data)
    transactions = pd.DataFrame({
        'Week': [1, 1, 2, 2],
        'CustomerID': [1, 2, 1, 3],
        'SKUID': [101, 102, 103, 104]
    })
    customers = pd.DataFrame({
        'CustomerID': [1, 2, 3],
        'State': ['CA', 'NY', 'TX']
    })
    skus = pd.DataFrame({
        'SKUID': [101, 102, 103, 104, 105]
    })
    
    # Generate training data
    training_data = generate_training_data(skus, customers, transactions, num_weeks=2)
    print(training_data)

In [ ]:
uv pip install dask 

## Changes 

---

### **Key Changes and Improvements**

1. **Multiple Rolling Windows**:
   - The `ROLLING_WINDOWS` constant is set to `[4, 8, 12]` weeks.
   - For each rolling feature (e.g., Customer_RollingAvgOrderValue, SKU_RollingTotalSales), the code computes three versions (e.g., `_4Weeks`, `_8Weeks`, `_12Weeks`).
   - This applies to:
     - `Customer_RollingAvgOrderValue_XWeeks`
     - `Customer_RollingUniqueSKUsPurchased_XWeeks`
     - `Customer_RollingPurchaseCount_XWeeks`
     - `SKU_RollingTotalSales_XWeeks`
     - `SKU_RollingPurchaseCount_XWeeks`
     - `Category_RollingTotalSales_XWeeks`
     - `Town_SKU_PurchaseCount_XWeeks`

2. **Trend-Based Feature**:
   - Added `Customer_PurchaseCount_Trend`, which computes the slope of purchase counts over the longest window (12 weeks) using linear regression (`scipy.stats.linregress`).
   - This captures whether a customer’s purchase frequency is increasing or decreasing, adding a dynamic signal to the feature set.

3. **Sparsity Handling**:
   - For rolling features, zero values are used when no transactions exist in the window (e.g., `Customer_RollingAvgOrderValue_4Weeks = 0` if no purchases).
   - In a real-world scenario, you could add smoothing (e.g., add a small constant or use exponential moving averages) to reduce the impact of sparse data.

4. **Consistency Across Training and Inference**:
   - The same feature computation logic is used for both training and inference, ensuring the model sees consistent features.
   - Features are computed using data only from weeks prior to the target week to prevent temporal leakage.

---

### **Potential Pitfalls and Mitigations**

1. **Increased Feature Dimensionality**:
   - **Issue**: Using multiple windows increases the number of features (e.g., three versions of each rolling feature), which could lead to overfitting or computational overhead.
   - **Mitigation**: Use feature selection techniques (e.g., permutation importance, L1 regularization) during model training to identify the most predictive windows. Alternatively, reduce dimensionality using PCA or feature aggregation (e.g., mean of rolling features across windows).

2. **Sparsity in Longer Windows**:
   - **Issue**: For customers or SKUs with infrequent purchases, longer windows (e.g., 12 weeks) may result in sparse or zero-valued features.
   - **Mitigation**: Apply smoothing (e.g., add a small constant or use exponential moving averages) or aggregate features at a higher level (e.g., Category instead of SKU). The code already sets zero for empty windows, but you could enhance this with smoothing logic.

3. **Computational Cost**:
   - **Issue**: Computing features for multiple windows increases processing time, especially for large datasets.
   - **Mitigation**: Optimize with vectorized operations or use distributed frameworks like Spark/Dask for large-scale data. Cache intermediate results (e.g., rolling aggregates) to avoid redundant calculations.

4. **Feature Correlation**:
   - **Issue**: Features from different windows (e.g., Customer_RollingPurchaseCount_4Weeks and _8Weeks) may be highly correlated, reducing their unique contribution.
   - **Mitigation**: Compute correlation matrices during EDA and drop highly correlated features, or use models robust to multicollinearity (e.g., tree-based models like XGBoost).

---

### **Additional Enhancement Suggestions**

1. **Feature Aggregation Across Windows**:
   - Compute ratios or differences between windows (e.g., `Customer_RollingPurchaseCount_4Weeks / Customer_RollingPurchaseCount_12Weeks`) to capture relative changes in behavior.
   - Example: Add to `compute_features`:
     ```python
     for window in ROLLING_WINDOWS:
         if feature_dict[f'Customer_RollingPurchaseCount_{window}Weeks'] > 0:
             feature_dict[f'PurchaseCount_Ratio_{window}Weeks_to_12Weeks'] = (
                 feature_dict[f'Customer_RollingPurchaseCount_{window}Weeks'] /
                 feature_dict[f'Customer_RollingPurchaseCount_12Weeks'] if feature_dict[f'Customer_RollingPurchaseCount_12Weeks'] > 0 else 1
             )
         else:
             feature_dict[f'PurchaseCount_Ratio_{window}Weeks_to_12Weeks'] = 0
     ```

2. **Exponential Moving Averages**:
   - Replace simple rolling averages with exponential moving averages to give more weight to recent weeks.
   - Example: Use `pandas.ewm` for `Customer_RollingAvgOrderValue_XWeeks`.

3. **Customer Segmentation**:
   - Add features based on customer lifecycle stages (e.g., new, loyal, at-risk) by clustering customers on RFM metrics and computing rolling features per segment.

4. **Dynamic Window Selection**:
   - During training, evaluate which window sizes are most predictive using feature importance or cross-validation, and use only the top-performing windows for inference.

---

### **How to Use the Updated Code**

1. **Dependencies**:
   - Install required libraries: `pip install pandas numpy scipy`
   - The `scipy.stats.linregress` function is used for the trend-based feature.

2. **Running the Code**:
   - Save the code to a file (e.g., `sku_prediction_data_generation_multi_windows.py`).
   - Run it: `python sku_prediction_data_generation_multi_windows.py`.
   - Outputs:
     - `training_data_multi_windows.csv`: Training data with multiple rolling window features.
     - `inference_data_multi_windows.csv`: Inference data with the same features.

3. **Adapting to Real Data**:
   - Replace `generate_synthetic_data` with your actual data loading logic.
   - Extend the feature set to include all features from the project summary (e.g., Segment_RollingTotalSales_XWeeks).
   - Adjust `ROLLING_WINDOWS` or add smoothing logic based on your data characteristics.

4. **Model Training**:
   - Use the generated training data to train a model (e.g., XGBoost, LightGBM).
   - Monitor feature importance to evaluate the contribution of each window size.
   - Use the inference data to predict purchase likelihood for Week 11.

---

This updated implementation enhances the Time-Series & Rolling Window features by incorporating multiple window sizes and a trend-based feature, addressing the project’s concern about optimal window sizes and improving the model’s ability to capture diverse temporal patterns. Let me know if you’d like to explore additional enhancements, such as specific feature aggregations or smoothing techniques!